# YOLO Training Template — Campus Hazard Detection

**One notebook per member.** Copy this to `train_yolo_member1.ipynb` etc.
and set `MEMBER` + `DATA_CONFIG` below. Each member trains ONE model over 5
classes (see `config/classes.yaml`). The report (§5, §8, §13) needs the
hyperparameters, training iterations, and evaluation metrics produced here.

In [ ]:
# Setup: run once. Install GPU torch FIRST, then ultralytics:
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
#   pip install ultralytics
import os, subprocess, sys
import torch
from ultralytics import YOLO

print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# Pretrained weights: this machine's SSL blocks ultralytics' auto-download, so
# fetch yolov8n.pt via the helper (curl --ssl-no-revoke) into the repo root.
subprocess.run([sys.executable, "download_weights.py", "n"], check=True)
WEIGHTS = "../../yolov8n.pt"   # download_weights.py saves to the repo root

MEMBER = "member1"  # <-- change per member: member1 | member2 | member3
DATA_CONFIG = f"../../config/{MEMBER}_data.yaml"
assert os.path.exists(DATA_CONFIG), DATA_CONFIG
print("Training", MEMBER, "from", DATA_CONFIG)

## Hyperparameters
Document every value you use — the report requires epochs, batch size, image
size, learning rate, optimizer, and augmentation settings. Record each
training *iteration* (initial, retrain, augmentation change, threshold tuning).

In [ ]:
model = YOLO(WEIGHTS)  # nano = fast + low VRAM; try yolov8s.pt for accuracy

# NOTE on 4 GB VRAM (RTX 3050): batch=8 at imgsz=640 fits comfortably.
# If you hit "CUDA out of memory", lower batch (4) or imgsz (512/416).
results = model.train(
    data=DATA_CONFIG,
    epochs=100,
    imgsz=640,
    batch=8,
    device=DEVICE,      # GPU if available, else CPU
    lr0=0.01,
    optimizer="auto",
    patience=20,        # early stopping
    project=f"../../runs/{MEMBER}",
    name="train",
    # augmentation (tune as a training iteration):
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0,
)

## Evaluation (assignment §13)
Report precision, recall, mAP@0.5, mAP@0.5:0.95, and the confusion matrix.
Ultralytics writes the confusion matrix + PR curves to the run folder.

In [ ]:
metrics = model.val(data=DATA_CONFIG)
print("mAP@0.5      :", metrics.box.map50)
print("mAP@0.5:0.95 :", metrics.box.map)
print("precision    :", metrics.box.mp)
print("recall       :", metrics.box.mr)
# Confusion matrix + curves saved under the run directory (see plots).

## Error analysis
Inspect false positives / false negatives, class confusion (especially between
overlap classes), and small-object misses. Note improvement attempts here for
the report's training-iteration table.

In [ ]:
# Export the best weights for the meta-classifier / mobile app.
# best = f"../../runs/{MEMBER}/train/weights/best.pt"
# model = YOLO(best)
# model.export(format="onnx")    # or format="tflite" for on-device Flutter
print("Remember to copy best.pt to models/ and share via Drive (git-ignored).")